# CLAP Zero Shot Audio Classification

Fresh Python3.11+ CPU kernel; pinned official CLAP weights and original ESC-50 dog clip, no API key. Use the source ten-row Hub-loading alternative instead of an unavailable local dataset path. All5 seconds of the exact source clip are processed; not a full2000-clip benchmark. Preserve both source candidate-label sets,48000Hz conversion and actual relative scores. Original code/transcripts remain in appendices. ESC-50 dataset is research/noncommercial; the dog example is in its ESC-10 subset. No synthetic audio or mock model outputs.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0", "transformers==4.57.6", "numpy==2.3.5", "datasets==3.6.0", "soundfile==0.13.1", "librosa==0.11.0", "gradio==6.3.0", "certifi==2026.10.23", "ipywidgets==8.1.7"], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
import certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()
import numpy as np
import torch
import transformers
import soundfile as sf
import librosa
from pathlib import Path
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

## Example 0

In [ ]:
print("Pinned Transformers, datasets, soundfile and librosa packages installed in the active kernel.")

## Example 1

In [ ]:
from datasets import load_dataset, Audio
DATA_REVISION = "e3e2a63ffff66b9a9735524551e3818e96af03ee"
dataset = load_dataset("ashraq/esc50", revision=DATA_REVISION, split="train[:10]", token=False, trust_remote_code=False)
original_dataset = dataset
print("Loaded source-example subset:", len(dataset), "records | revision:", DATA_REVISION)

## Example 2

In [ ]:
import hashlib
audio_sample = dataset[0]
original_audio = audio_sample["audio"]["array"].copy()
original_rate = audio_sample["audio"]["sampling_rate"]
raw_audio = original_dataset.cast_column("audio", Audio(decode=False))[0]["audio"]
audio_sha = hashlib.sha256(raw_audio["bytes"]).hexdigest()
print({key: value for key, value in audio_sample.items() if key != "audio"})
print("Audio samples / rate / duration:", len(original_audio), original_rate, len(original_audio)/original_rate)
print("Original WAV SHA-256:", audio_sha)

## Example 3

In [ ]:
from IPython.display import Audio as IPythonAudio, display
display(IPythonAudio(original_audio, rate=original_rate))
print("Notebook audio player:", len(original_audio)/original_rate, "seconds at", original_rate, "Hz")

## Example 4

In [ ]:
from transformers import pipeline
MODEL_ID = "laion/clap-htsat-unfused"
MODEL_REVISION = "8fa0f1c6d0433df6e97c127f64b2a1d6c0dcda8a"
zero_shot_classifier = pipeline(task="zero-shot-audio-classification", model=MODEL_ID, revision=MODEL_REVISION, token=False, device=-1)
zero_shot_classifier.model.eval()
print("Loaded:", MODEL_ID, "| revision:", MODEL_REVISION)

## Example 5

In [ ]:
print(zero_shot_classifier.feature_extractor.sampling_rate)

## Example 6

In [ ]:
print(audio_sample["audio"]["sampling_rate"])

## Example 7

In [ ]:
dataset = dataset.cast_column("audio", Audio(sampling_rate=48_000))
audio_sample = dataset[0]
resampled_audio = audio_sample["audio"]["array"]
print("Resampled audio:", len(resampled_audio), "samples at", audio_sample["audio"]["sampling_rate"], "Hz")
print("Duration:", len(resampled_audio)/48_000, "seconds")

## Example 8

In [ ]:
candidate_labels = ["Sound of a dog", "Sound of vacuum cleaner"]
print(candidate_labels)

## Example 9

In [ ]:
relevant_labels = candidate_labels.copy()
relevant_results = zero_shot_classifier(resampled_audio, candidate_labels=candidate_labels)
print(relevant_results)

## Example 10

In [ ]:
candidate_labels = ["Sound of a child crying", "Sound of vacuum cleaner", "Sound of a bird singing", "Sound of an airplane"]
print(candidate_labels)

## Example 11

In [ ]:
unrelated_labels = candidate_labels.copy()
unrelated_results = zero_shot_classifier(resampled_audio, candidate_labels=candidate_labels)
print(unrelated_results)

## Independent regression checks

In [ ]:
assert len(dataset) == 10 and audio_sample["filename"] == "1-100032-A-0.wav"
assert original_rate == 44100 and len(original_audio) == 220500
assert audio_sample["audio"]["sampling_rate"] == 48000 and abs(len(resampled_audio) - 240000) <= 1
assert abs(len(resampled_audio)/48000 - 5.0) <= 1/48000
assert np.isfinite(resampled_audio).all() and not zero_shot_classifier.model.training
independent_resample = librosa.resample(original_audio, orig_sr=44100, target_sr=48000, res_type="soxr_hq")
np.testing.assert_allclose(resampled_audio, independent_resample, atol=1e-7)
for labels, results in [(relevant_labels, relevant_results), (unrelated_labels, unrelated_results)]:
    assert {v["label"] for v in results} == set(labels)
    assert all(np.isfinite(v["score"]) and 0 <= v["score"] <= 1 for v in results)
    np.testing.assert_allclose(sum(v["score"] for v in results), 1, atol=1e-6)
    audio_inputs = zero_shot_classifier.feature_extractor([resampled_audio], sampling_rate=48000, return_tensors="pt")
    text_inputs = zero_shot_classifier.tokenizer([f"This is a sound of {label}." for label in labels], padding=True, return_tensors="pt")
    with torch.inference_mode():
        logits = zero_shot_classifier.model(**audio_inputs, **text_inputs).logits_per_audio[0].numpy().astype("float64")
    exp = np.exp(logits - logits.max())
    reference = exp / exp.sum()
    actual = {v["label"]:v["score"] for v in results}
    np.testing.assert_allclose([actual[label] for label in labels], reference, atol=2e-6, rtol=1e-5)
    reversed_results = zero_shot_classifier(resampled_audio, candidate_labels=labels[::-1])
    np.testing.assert_allclose([next(v["score"] for v in reversed_results if v["label"]==label) for label in labels], reference, atol=2e-6, rtol=1e-5)
    print("Verified label set:", labels)
assert relevant_results[0]["label"] == "Sound of a dog"
assert all(p.grad is None for p in zero_shot_classifier.model.parameters())
print("Complete five-second source audio, resampling, both real model classifications, independent softmax and label-permutation checks passed.")

## Original source appendix

Full original blocks from raw source SHA-256 9d537f85092b3b4b94e09a716b2488a41a3289d617bc51004053069560ebe473. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
!pip install transformers
    !pip install datasets
    !pip install soundfile
    !pip install librosa
```

### Original block 2

```text
from datasets import load_dataset, load_from_disk

# This dataset is a collection of different sounds of 5 seconds
# dataset = load_dataset("ashraq/esc50",
#                       split="train[0:10]")
dataset = load_from_disk("./models/ashraq/esc50/train")
```

### Original block 3

```text
audio_sample = dataset[0]
audio_sample
```

### Original block 4

```text
from IPython.display import Audio as IPythonAudio
IPythonAudio(audio_sample["audio"]["array"],
             rate=audio_sample["audio"]["sampling_rate"])
```

### Original block 5

```text
from transformers import pipeline
zero_shot_classifier = pipeline(
    task="zero-shot-audio-classification",
    model="./models/laion/clap-htsat-unfused")
```

### Original block 6

```text
zero_shot_classifier.feature_extractor.sampling_rate
```

### Original block 7

```text
audio_sample["audio"]["sampling_rate"]
```

### Original block 8

```text
from datasets import Audio
dataset = dataset.cast_column(
    "audio",
     Audio(sampling_rate=48_000))

audio_sample = dataset[0]
audio_sample
```

### Original block 9

```text
candidate_labels = ["Sound of a dog",
                    "Sound of vacuum cleaner"]
```

### Original block 10

```text
zero_shot_classifier(audio_sample["audio"]["array"],
                     candidate_labels=candidate_labels)
```

### Original block 11

```text
candidate_labels = ["Sound of a child crying",
                    "Sound of vacuum cleaner",
                    "Sound of a bird singing",
                    "Sound of an airplane"]
```

### Original block 12

```text
zero_shot_classifier(audio_sample["audio"]["array"],
                     candidate_labels=candidate_labels)
```

## Original source outputs

{‘filename’: ‘1–100032-A-0.wav’, ‘fold’: 1, ‘target’: 0, ‘category’: ‘dog’, ‘esc10’: True, ‘src_file’: 100032, ‘take’: ‘A’, ‘audio’: {‘path’: None, ‘array’: array([0., 0., 0., …, 0., 0., 0.]), ‘sampling_rate’: 44100}}

48000

44100

{‘filename’: ‘1–100032-A-0.wav’, ‘fold’: 1, ‘target’: 0, ‘category’: ‘dog’, ‘esc10’: True, ‘src_file’: 100032, ‘take’: ‘A’, ‘audio’: {‘path’: None, ‘array’: array([0., 0., 0., …, 0., 0., 0.]), ‘sampling_rate’: 48000}}

[{‘score’: 0.9985589385032654, ‘label’: ‘Sound of a dog’}, {‘score’: 0.0014411123702302575, ‘label’: ‘Sound of vacuum cleaner’}]

[{‘score’: 0.6172530055046082, ‘label’: ‘Sound of a bird singing’}, {‘score’: 0.21602635085582733, ‘label’: ‘Sound of vacuum cleaner’}, {‘score’: 0.12547191977500916, ‘label’: ‘Sound of an airplane’}, {‘score’: 0.04124866798520088, ‘label’: ‘Sound of a child crying’}]